# XDATCAR 数据分析：RDF计算和反应时间点分析

本notebook用于分析VASP的XDATCAR文件，包括：
1. 读取和解析XDATCAR文件
2. 计算径向分布函数（RDF）
3. 分析反应发生的时间点
4. 可视化结果


In [ ]:
# 设置matplotlib在Jupyter notebook中内联显示图像
%matplotlib inline
# 如果需要交互式绘图，可以使用: %matplotlib widget (需要安装ipympl)

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy import signal
import warnings
import matplotlib.font_manager as fm
import os
warnings.filterwarnings('ignore')

# 设置matplotlib的显示参数
plt.rcParams['figure.dpi'] = 100  # 设置图像分辨率
plt.rcParams['savefig.dpi'] = 300  # 保存图像时的分辨率
plt.rcParams['figure.figsize'] = (10, 6)  # 默认图像大小

# 设置中文字体 - 自动检测系统可用的中文字体
def setup_chinese_font(clear_cache=False):
    """
    设置matplotlib的中文字体
    
    参数:
        clear_cache: 是否清除matplotlib字体缓存（如果字体不生效可以尝试设为True）
    """
    if clear_cache:
        # 清除matplotlib字体缓存
        try:
            cache_dir = fm.get_cachedir()
            import shutil
            if os.path.exists(cache_dir):
                shutil.rmtree(cache_dir)
                print(f"已清除字体缓存: {cache_dir}")
                # 重新构建字体缓存
                fm._rebuild()
        except Exception as e:
            print(f"清除缓存时出错: {e}")
    
    # 常见的中文字体名称列表（按优先级排序）- 排除不支持中文的字体
    chinese_fonts = [
        'SimHei',              # Windows黑体
        'Microsoft YaHei',     # Windows微软雅黑
        'WenQuanYi Micro Hei', # Linux文泉驿微米黑
        'WenQuanYi Zen Hei',   # Linux文泉驿正黑
        'Noto Sans CJK SC',    # Google Noto字体
        'Noto Sans SC',        # Google Noto字体（简化名）
        'Source Han Sans CN',  # Adobe思源黑体
        'Droid Sans Fallback', # Droid字体（支持中文）
        'Droid Sans',          # Droid字体（可能支持中文）
        'STHeiti',             # Mac黑体
        'STSong',              # Mac宋体
        'Arial Unicode MS',    # 包含中文的Arial
    ]
    
    # 明确排除不支持中文的字体
    excluded_fonts = ['DejaVu Sans', 'DejaVu Serif', 'Arial', 'Times New Roman', 
                      'Courier New', 'Helvetica', 'Liberation Sans']
    
    # 获取系统所有可用字体
    try:
        available_fonts = [f.name for f in fm.fontManager.ttflist]
    except:
        # 如果获取字体列表失败，尝试重建
        fm._rebuild()
        available_fonts = [f.name for f in fm.fontManager.ttflist]
    
    # 测试字体是否支持中文的函数
    def test_chinese_support(font_name):
        """测试字体是否支持中文显示"""
        try:
            # 获取字体文件路径
            font_files = [f.fname for f in fm.fontManager.ttflist if f.name == font_name]
            if not font_files:
                return False
            
            # 方法1: 检查字体文件是否包含CJK字符（通过文件名和字体信息）
            font_file = font_files[0]
            font_info = [f for f in fm.fontManager.ttflist if f.name == font_name][0]
            
            # 检查字体名称是否包含中文字体特征
            if any(keyword in font_name for keyword in ['CJK', 'Chinese', 'SC', 'CN', 'Hei', 'Song', 'WenQuanYi', 'Noto']):
                return True
            
            # 方法2: 尝试实际渲染中文文本
            try:
                font_prop = fm.FontProperties(fname=font_file)
                # 使用非交互式后端测试
                import matplotlib
                old_backend = matplotlib.get_backend()
                matplotlib.use('Agg')  # 使用非交互式后端
                fig, ax = plt.subplots(figsize=(1, 1))
                text_obj = ax.text(0.5, 0.5, '测试', fontproperties=font_prop, fontsize=12)
                # 尝试获取文本的边界框
                try:
                    fig.canvas.draw()
                    bbox = text_obj.get_window_extent()
                    plt.close(fig)
                    matplotlib.use(old_backend)  # 恢复原来的后端
                    # 如果文本宽度大于0，说明字体可能支持中文
                    return bbox.width > 0
                except:
                    plt.close(fig)
                    matplotlib.use(old_backend)
                    # 如果无法测试，但字体名称看起来像中文字体，返回True
                    return any(keyword in font_name.lower() for keyword in ['cjk', 'chinese', 'hei', 'song'])
            except:
                # 如果测试失败，但字体名称包含中文字体特征，返回True
                return any(keyword in font_name for keyword in ['CJK', 'Chinese', 'SC', 'CN', 'Hei', 'Song', 'WenQuanYi'])
        except:
            return False
    
    # 查找第一个可用的且支持中文的字体
    font_found = None
    for font in chinese_fonts:
        if font in available_fonts and font not in excluded_fonts:
            # 测试字体是否真正支持中文
            if test_chinese_support(font):
                font_found = font
                print(f"✓ 找到并验证中文字体: {font}")
                break
            else:
                print(f"  ⚠ {font} 在系统中但可能不支持中文，跳过")
    
    if not font_found:
        # 方法1: 尝试通过字体文件路径直接查找Droid Sans Fallback
        try:
            import subprocess
            result = subprocess.run(['fc-list', ':lang=zh'], 
                                  capture_output=True, text=True, timeout=5)
            if result.returncode == 0 and result.stdout.strip():
                # 查找Droid Sans Fallback
                for line in result.stdout.strip().split('\n'):
                    if 'DroidSansFallback' in line or 'Droid Sans Fallback' in line:
                        # 提取字体文件路径
                        font_path = line.split(':')[0]
                        if os.path.exists(font_path):
                            # 直接通过文件路径创建字体属性
                            font_prop_direct = fm.FontProperties(fname=font_path)
                            # 测试字体
                            try:
                                import matplotlib
                                old_backend = matplotlib.get_backend()
                                matplotlib.use('Agg')
                                fig, ax = plt.subplots(figsize=(1, 1))
                                ax.text(0.5, 0.5, '测试', fontproperties=font_prop_direct, fontsize=12)
                                fig.canvas.draw()
                                plt.close(fig)
                                matplotlib.use(old_backend)
                                font_found = 'Droid Sans Fallback'
                                print(f"✓ 通过字体文件路径找到中文字体: Droid Sans Fallback")
                                print(f"  字体文件: {font_path}")
                                # 设置字体
                                plt.rcParams['font.sans-serif'] = ['Droid Sans Fallback'] + plt.rcParams['font.sans-serif']
                                break
                            except:
                                matplotlib.use(old_backend)
                                plt.close(fig)
        except:
            pass
        
        # 方法2: 如果没有找到，尝试查找包含CJK或中文的字体
        if not font_found:
            cjk_keywords = ['CJK', 'Chinese', 'SC', 'CN', 'Hei', 'Song', 'Ming', 'Kai', 'WenQuanYi', 'Droid']
            cjk_fonts = []
            for f in available_fonts:
                if f not in excluded_fonts and any(keyword in f for keyword in cjk_keywords):
                    cjk_fonts.append(f)
            
            # 测试这些字体是否支持中文
            for font in cjk_fonts:
                if test_chinese_support(font):
                    font_found = font
                    print(f"✓ 找到并验证中文字体: {font_found}")
                    break
        
        if not font_found:
            print("="*70)
            print("⚠ 警告: 未找到支持中文的字体！")
            print("="*70)
            print(f"   系统共有 {len(available_fonts)} 个可用字体")
            print("   可用字体示例（前30个）:", available_fonts[:30])
            print("\n   ⚠ 重要提示: 需要安装中文字体才能正确显示中文！")
            print("\n   安装方法（在终端中执行）:")
            print("   - CentOS/RHEL: sudo yum install wqy-microhei-fonts")
            print("   - Ubuntu/Debian: sudo apt-get install fonts-wqy-microhei")
            print("   - 或下载Noto字体: https://www.google.com/get/noto/")
            print("\n   检查系统是否已安装中文字体（在终端执行）:")
            print("   - fc-list :lang=zh  # 查看已安装的中文字体")
            print("   - fc-list | grep -i cjk  # 查看CJK字体")
            print("\n   安装后请:")
            print("   1. 重启Jupyter kernel")
            print("   2. 重新运行此cell")
            print("="*70)
            return None, None
    
    # 设置找到的字体
    if font_found:
        plt.rcParams['font.sans-serif'] = [font_found] + plt.rcParams['font.sans-serif']
    
    # 解决负号显示问题
    plt.rcParams['axes.unicode_minus'] = False
    
    # 最终测试中文显示
    if font_found:
        try:
            fig, ax = plt.subplots(figsize=(2, 1))
            ax.text(0.5, 0.5, '测试中文显示', fontsize=14, ha='center', va='center')
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.axis('off')
            # 保存测试图像
            plt.savefig('font_test.png', dpi=100, bbox_inches='tight')
            plt.close(fig)
            print("✓ 中文字体设置成功，已生成测试图像 font_test.png")
        except Exception as e:
            print(f"⚠ 字体测试时出错: {e}")
    
    # 获取字体属性对象（用于在绘图时显式指定）
    font_prop = None
    if font_found:
        try:
            # 方法1: 如果font_found是'Droid Sans Fallback'，尝试通过文件路径直接加载
            if 'Droid' in font_found:
                try:
                    import subprocess
                    result = subprocess.run(['fc-list', ':lang=zh'], 
                                          capture_output=True, text=True, timeout=5)
                    if result.returncode == 0:
                        for line in result.stdout.strip().split('\n'):
                            if 'DroidSansFallback' in line:
                                font_path = line.split(':')[0]
                                if os.path.exists(font_path):
                                    font_prop = fm.FontProperties(fname=font_path)
                                    print(f"✓ 通过文件路径加载字体: {font_path}")
                                    break
                except:
                    pass
            
            # 方法2: 通过字体名称创建FontProperties
            if font_prop is None:
                font_prop = fm.FontProperties(family=font_found)
                # 测试字体是否可用
                test_fig, test_ax = plt.subplots(figsize=(1, 1))
                test_ax.text(0.5, 0.5, '测试', fontproperties=font_prop, fontsize=12)
                plt.close(test_fig)
        except Exception as e1:
            try:
                # 方法3: 通过字体文件路径创建
                font_files = [f.fname for f in fm.fontManager.ttflist if f.name == font_found]
                if font_files:
                    font_prop = fm.FontProperties(fname=font_files[0])
                    # 测试字体是否可用
                    test_fig, test_ax = plt.subplots(figsize=(1, 1))
                    test_ax.text(0.5, 0.5, '测试', fontproperties=font_prop, fontsize=12)
                    plt.close(test_fig)
            except Exception as e2:
                print(f"⚠ 创建字体属性对象时出错: {e1}, {e2}")
                # 方法4: 直接使用字体名称字符串（matplotlib会自动查找）
                font_prop = font_found
    
    return font_found, font_prop

# 首先检查系统是否有中文字体（可选，用于诊断）
def check_system_chinese_fonts():
    """检查系统中是否有中文字体"""
    try:
        import subprocess
        result = subprocess.run(['fc-list', ':lang=zh'], 
                              capture_output=True, text=True, timeout=5)
        if result.returncode == 0 and result.stdout.strip():
            fonts = result.stdout.strip().split('\n')
            print(f"✓ 系统检测到 {len(fonts)} 个中文字体:")
            for font in fonts[:10]:  # 只显示前10个
                print(f"   {font}")
            if len(fonts) > 10:
                print(f"   ... 还有 {len(fonts)-10} 个字体")
            return True
        else:
            print("⚠ 系统未检测到中文字体（fc-list命令可能不可用或未安装字体）")
            return False
    except FileNotFoundError:
        print("⚠ fc-list命令不可用，无法检查系统字体")
        return None
    except Exception as e:
        print(f"⚠ 检查字体时出错: {e}")
        return None

# 检查系统字体（可选）
# check_system_chinese_fonts()

# 设置字体
# 注意：如果第一次运行找不到字体，尝试将clear_cache设为True来清除matplotlib字体缓存
chinese_font_name, chinese_font_prop = setup_chinese_font(clear_cache=True)  # 清除缓存以确保识别Droid字体

# 全局设置字体属性（确保所有绘图都使用中文字体）
if chinese_font_name:
    if chinese_font_prop:
        try:
            if hasattr(chinese_font_prop, 'get_name'):
                plt.rcParams['font.family'] = chinese_font_prop.get_name()
                print(f"✓ 已设置全局字体: {chinese_font_prop.get_name()}")
            else:
                plt.rcParams['font.family'] = chinese_font_name
                print(f"✓ 已设置全局字体: {chinese_font_name}")
        except:
            plt.rcParams['font.family'] = chinese_font_name
            print(f"✓ 已设置全局字体: {chinese_font_name}")
    else:
        plt.rcParams['font.family'] = chinese_font_name
        print(f"✓ 已设置全局字体: {chinese_font_name}")
else:
    print("⚠ 警告: 未找到中文字体，图像中的中文可能无法正确显示！")
    print("   请按照上面的提示安装中文字体后重新运行此cell。")

# 确保unicode_minus设置正确
plt.rcParams['axes.unicode_minus'] = False

# 创建一个辅助函数来获取字体属性（用于绘图）
def get_font_prop():
    """获取中文字体属性，如果不存在则返回None"""
    if 'chinese_font_prop' in globals() and chinese_font_prop:
        return chinese_font_prop
    return None

# 创建一个辅助函数来设置文本的字体属性
def set_text_font(text_obj, font_prop):
    """为文本对象设置字体属性"""
    if font_prop:
        if isinstance(font_prop, str):
            text_obj.set_fontfamily(font_prop)
        else:
            text_obj.set_fontproperties(font_prop)

# 创建一个辅助函数来获取字体参数字典
def get_font_kwargs(font_prop):
    """返回用于matplotlib文本函数的字体参数字典"""
    if font_prop:
        if isinstance(font_prop, str):
            return {'fontfamily': font_prop}
        else:
            return {'fontproperties': font_prop}
    return {}

print("库导入完成！")

# 测试图像显示是否正常
print("\n测试图像显示...")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([1, 2, 3, 4], [1, 4, 2, 3], 'o-', label='测试数据')
ax.set_xlabel('X轴', fontsize=12)
ax.set_ylabel('Y轴', fontsize=12)
ax.set_title('图像显示测试', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("✓ 如果上方显示了图像，说明图像显示功能正常！")


清除缓存时出错: module 'matplotlib.font_manager' has no attribute 'get_cachedir'
✓ 找到并验证中文字体: Droid Sans Fallback
✓ 中文字体设置成功，已生成测试图像 font_test.png
✓ 通过文件路径加载字体: /usr/share/fonts/google-droid/DroidSansFallback.ttf
✓ 已设置全局字体: Droid Sans Fallback
库导入完成！


## 1. 读取XDATCAR文件


In [2]:
def read_xdatcar(filename):
    """
    读取XDATCAR文件
    
    返回:
        lattice: 晶格向量 (3x3)
        elements: 元素列表
        n_atoms: 各元素原子数列表
        configurations: 所有构型的原子坐标列表
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # 读取系统名称
    system_name = lines[0].strip()
    
    # 读取缩放因子
    scale = float(lines[1].strip())
    
    # 读取晶格向量
    lattice = np.zeros((3, 3))
    for i in range(3):
        lattice[i] = [float(x) for x in lines[2+i].split()]
    lattice *= scale
    
    # 读取元素和原子数
    elements = lines[5].split()
    n_atoms = [int(x) for x in lines[6].split()]
    total_atoms = sum(n_atoms)
    
    print(f"系统名称: {system_name}")
    print(f"元素: {elements}")
    print(f"各元素原子数: {n_atoms}")
    print(f"总原子数: {total_atoms}")
    print(f"晶格向量:\n{lattice}")
    
    # 读取所有构型
    configurations = []
    i = 7
    while i < len(lines):
        if 'Direct configuration=' in lines[i]:
            config_num = int(lines[i].split('=')[1].strip())
            coords = []
            i += 1
            for j in range(total_atoms):
                if i + j < len(lines):
                    coords.append([float(x) for x in lines[i+j].split()])
            if len(coords) == total_atoms:
                coords = np.array(coords)
                # 将分数坐标转换为笛卡尔坐标
                cartesian = coords @ lattice
                configurations.append({
                    'config_num': config_num,
                    'fractional': coords,
                    'cartesian': cartesian
                })
            i += total_atoms
        else:
            i += 1
    
    print(f"\n成功读取 {len(configurations)} 个构型")
    return lattice, elements, n_atoms, configurations

# 读取文件
filename = 'XDATCAR'
lattice, elements, n_atoms, configs = read_xdatcar(filename)


系统名称: unknown system
元素: ['Na', 'Sb', 'S']
各元素原子数: [96, 12, 48]
总原子数: 156
晶格向量:
[[12.624161  0.        0.      ]
 [ 4.208054 11.902173  0.      ]
 [ 0.        0.       28.189447]]

成功读取 5000 个构型


## 2. 计算径向分布函数（RDF）


In [3]:
def calculate_rdf(coords1_cart, coords2_cart, lattice, r_max=10.0, dr=0.1):
    """
    计算两个原子组之间的径向分布函数
    
    参数:
        coords1_cart: 第一组原子的笛卡尔坐标
        coords2_cart: 第二组原子的笛卡尔坐标
        lattice: 晶格向量
        r_max: 最大距离
        dr: 距离间隔
    
    返回:
        r: 距离数组
        g_r: RDF值数组
    """
    # 使用最小镜像约定计算距离
    n1 = len(coords1_cart)
    n2 = len(coords2_cart)
    
    # 计算晶格向量的逆矩阵（用于将笛卡尔坐标转换为分数坐标）
    lattice_inv = np.linalg.inv(lattice)
    
    # 计算所有原子对之间的距离
    distances = []
    
    for coord1 in coords1_cart:
        for coord2 in coords2_cart:
            # 计算笛卡尔坐标差
            cart_diff = coord1 - coord2
            
            # 转换为分数坐标差
            frac_diff = cart_diff @ lattice_inv
            
            # 应用周期性边界条件（将分数坐标差限制在[-0.5, 0.5]）
            frac_diff = frac_diff - np.round(frac_diff)
            
            # 转换回笛卡尔坐标差（最小镜像约定）
            cart_diff_min = frac_diff @ lattice
            
            # 计算距离
            dist = np.linalg.norm(cart_diff_min)
            if dist > 0.01:  # 排除自身
                distances.append(dist)
    
    distances = np.array(distances)
    
    # 计算RDF
    r = np.arange(dr, r_max, dr)
    g_r = np.zeros_like(r)
    
    volume = np.abs(np.linalg.det(lattice))
    density = n2 / volume
    
    for i, r_val in enumerate(r):
        # 计算在r到r+dr范围内的原子对数
        count = np.sum((distances >= r_val) & (distances < r_val + dr))
        # RDF公式: g(r) = n(r) / (4πr²ρdr)
        if r_val > 0:
            g_r[i] = count / (4 * np.pi * r_val**2 * density * dr * n1)
    
    return r, g_r

def calculate_rdf_for_pair(configs, elements, n_atoms, elem1_idx, elem2_idx, 
                           r_max=10.0, dr=0.1, step=10):
    """
    计算特定元素对之间的RDF随时间的变化
    
    参数:
        configs: 所有构型
        elements: 元素列表
        n_atoms: 各元素原子数
        elem1_idx: 第一个元素的索引
        elem2_idx: 第二个元素的索引
        r_max: 最大距离
        dr: 距离间隔
        step: 每隔多少个构型计算一次（用于加速）
    
    返回:
        r: 距离数组
        rdf_time: RDF随时间的变化 (n_configs x n_r)
    """
    # 确定原子索引范围
    start_idx1 = sum(n_atoms[:elem1_idx])
    end_idx1 = start_idx1 + n_atoms[elem1_idx]
    
    start_idx2 = sum(n_atoms[:elem2_idx])
    end_idx2 = start_idx2 + n_atoms[elem2_idx]
    
    rdf_time = []
    r = None
    
    print(f"计算 {elements[elem1_idx]}-{elements[elem2_idx]} 对的RDF...")
    
    for i in range(0, len(configs), step):
        config = configs[i]
        coords = config['cartesian']
        
        coords1 = coords[start_idx1:end_idx1]
        coords2 = coords[start_idx2:end_idx2]
        
        r, g_r = calculate_rdf(coords1, coords2, lattice, r_max=r_max, dr=dr)
        rdf_time.append(g_r)
        
        if (i + 1) % 100 == 0:
            print(f"  已处理 {i+1}/{len(configs)} 个构型")
    
    rdf_time = np.array(rdf_time)
    return r, rdf_time

print("RDF计算函数定义完成！")


RDF计算函数定义完成！


In [4]:
# 计算所有元素对的RDF
# 可以根据需要选择要分析的元素对
print("可用的元素对:")
for i, elem1 in enumerate(elements):
    for j, elem2 in enumerate(elements):
        if i <= j:
            print(f"  {i}-{j}: {elem1}-{elem2}")

# 计算几个重要的元素对的RDF
# 例如：Na-S, Sb-S, Na-Sb等
rdf_results = {}

# 计算Na-S的RDF
if 'Na' in elements and 'S' in elements:
    na_idx = elements.index('Na')
    s_idx = elements.index('S')
    r, rdf_na_s = calculate_rdf_for_pair(configs, elements, n_atoms, na_idx, s_idx, 
                                         r_max=8.0, dr=0.05, step=5)
    rdf_results['Na-S'] = {'r': r, 'rdf': rdf_na_s}

# 计算Sb-S的RDF
if 'Sb' in elements and 'S' in elements:
    sb_idx = elements.index('Sb')
    s_idx = elements.index('S')
    r, rdf_sb_s = calculate_rdf_for_pair(configs, elements, n_atoms, sb_idx, s_idx, 
                                        r_max=8.0, dr=0.05, step=5)
    rdf_results['Sb-S'] = {'r': r, 'rdf': rdf_sb_s}

# 计算Na-Sb的RDF
if 'Na' in elements and 'Sb' in elements:
    na_idx = elements.index('Na')
    sb_idx = elements.index('Sb')
    r, rdf_na_sb = calculate_rdf_for_pair(configs, elements, n_atoms, na_idx, sb_idx, 
                                         r_max=8.0, dr=0.05, step=5)
    rdf_results['Na-Sb'] = {'r': r, 'rdf': rdf_na_sb}

print(f"\n已计算 {len(rdf_results)} 个元素对的RDF")


可用的元素对:
  0-0: Na-Na
  0-1: Na-Sb
  0-2: Na-S
  1-1: Sb-Sb
  1-2: Sb-S
  2-2: S-S
计算 Na-S 对的RDF...
计算 Sb-S 对的RDF...
计算 Na-Sb 对的RDF...

已计算 3 个元素对的RDF


## 3. 可视化RDF结果


In [5]:
# 绘制RDF随时间的变化（热图）
fig, axes = plt.subplots(len(rdf_results), 1, figsize=(12, 4*len(rdf_results)))
if len(rdf_results) == 1:
    axes = [axes]

# 确保使用中文字体
font_prop = get_font_prop()
font_kwargs = get_font_kwargs(font_prop)

for idx, (pair_name, data) in enumerate(rdf_results.items()):
    r = data['r']
    rdf = data['rdf']
    
    # 创建热图
    im = axes[idx].imshow(rdf, aspect='auto', origin='lower', 
                         extent=[r[0], r[-1], 0, len(rdf)], 
                         cmap='viridis', interpolation='bilinear')
    
    # 显式设置字体属性
    axes[idx].set_xlabel('距离 r (Å)', fontsize=12, **font_kwargs)
    axes[idx].set_ylabel('构型编号', fontsize=12, **font_kwargs)
    axes[idx].set_title(f'{pair_name} 对的RDF随时间变化', fontsize=14, fontweight='bold', **font_kwargs)
    cbar = plt.colorbar(im, ax=axes[idx])
    cbar.set_label('g(r)', **font_kwargs)

plt.tight_layout()
plt.savefig('RDF_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


<Figure size 1200x1200 with 6 Axes>

In [6]:
# 绘制不同时间点的RDF曲线
fig, axes = plt.subplots(len(rdf_results), 1, figsize=(10, 4*len(rdf_results)))
if len(rdf_results) == 1:
    axes = [axes]

time_points = [0, len(rdf_results[list(rdf_results.keys())[0]]['rdf'])//4, 
               len(rdf_results[list(rdf_results.keys())[0]]['rdf'])//2,
               3*len(rdf_results[list(rdf_results.keys())[0]]['rdf'])//4,
               len(rdf_results[list(rdf_results.keys())[0]]['rdf'])-1]

colors = plt.cm.viridis(np.linspace(0, 1, len(time_points)))

# 确保使用中文字体
font_prop = get_font_prop()
font_kwargs = get_font_kwargs(font_prop)

for idx, (pair_name, data) in enumerate(rdf_results.items()):
    r = data['r']
    rdf = data['rdf']
    
    for i, t in enumerate(time_points):
        if t < len(rdf):
            axes[idx].plot(r, rdf[t], label=f'构型 {t*5+1}', color=colors[i], linewidth=2)
    
    # 显式设置字体属性
    axes[idx].set_xlabel('距离 r (Å)', fontsize=12, **font_kwargs)
    axes[idx].set_ylabel('g(r)', fontsize=12, **font_kwargs)
    axes[idx].set_title(f'{pair_name} 对的RDF在不同时间点', fontsize=14, fontweight='bold', **font_kwargs)
    if font_prop:
        axes[idx].legend(prop=font_prop if not isinstance(font_prop, str) else None)
    else:
        axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('RDF_curves.png', dpi=300, bbox_inches='tight')
plt.show()


<Figure size 1000x1200 with 3 Axes>

## 4. 分析反应发生的时间点


In [7]:
def analyze_reaction_time(rdf_results, r_range=(0, 3.0)):
    """
    通过分析RDF的变化来识别反应发生的时间点
    
    参数:
        rdf_results: RDF结果字典
        r_range: 要分析的距离范围（通常是最短键长范围）
    
    返回:
        reaction_times: 各元素对的反应时间点
    """
    reaction_times = {}
    
    for pair_name, data in rdf_results.items():
        r = data['r']
        rdf = data['rdf']
        
        # 找到r_range范围内的RDF
        mask = (r >= r_range[0]) & (r <= r_range[1])
        rdf_short = rdf[:, mask]
        
        # 计算每个时间点的积分（配位数）
        coordination = np.sum(rdf_short, axis=1) * (r[1] - r[0])
        
        # 计算配位数的变化率
        coordination_diff = np.diff(coordination)
        
        # 使用滑动窗口平滑
        window_size = min(50, len(coordination_diff) // 10)
        if window_size > 1:
            coordination_diff_smooth = np.convolve(coordination_diff, 
                                                   np.ones(window_size)/window_size, 
                                                   mode='same')
        else:
            coordination_diff_smooth = coordination_diff
        
        # 找到变化率最大的点（可能是反应发生点）
        # 也可以找配位数突然变化的地方
        threshold = np.std(coordination_diff_smooth) * 2
        
        # 找到显著变化点
        significant_changes = np.where(np.abs(coordination_diff_smooth) > threshold)[0]
        
        reaction_times[pair_name] = {
            'coordination': coordination,
            'coordination_diff': coordination_diff_smooth,
            'significant_changes': significant_changes,
            'first_change': significant_changes[0] if len(significant_changes) > 0 else None
        }
        
        print(f"\n{pair_name} 对:")
        print(f"  平均配位数: {np.mean(coordination):.2f}")
        print(f"  配位数变化范围: [{np.min(coordination):.2f}, {np.max(coordination):.2f}]")
        if len(significant_changes) > 0:
            print(f"  发现 {len(significant_changes)} 个显著变化点")
            print(f"  第一个显著变化发生在构型: {significant_changes[0]*5+1}")
        else:
            print(f"  未发现显著变化")
    
    return reaction_times

# 分析反应时间点
reaction_times = analyze_reaction_time(rdf_results, r_range=(0, 3.5))



Na-S 对:
  平均配位数: 2.58
  配位数变化范围: [1.98, 2.81]
  发现 40 个显著变化点
  第一个显著变化发生在构型: 1

Sb-S 对:
  平均配位数: 2.26
  配位数变化范围: [1.27, 4.81]
  发现 114 个显著变化点
  第一个显著变化发生在构型: 141

Na-Sb 对:
  平均配位数: 1.27
  配位数变化范围: [0.46, 1.81]
  发现 71 个显著变化点
  第一个显著变化发生在构型: 1


In [8]:
# 可视化配位数随时间的变化
fig, axes = plt.subplots(len(reaction_times), 1, figsize=(12, 4*len(reaction_times)))
if len(reaction_times) == 1:
    axes = [axes]

# 确保使用中文字体
font_prop = get_font_prop()
font_kwargs = get_font_kwargs(font_prop)

for idx, (pair_name, data) in enumerate(reaction_times.items()):
    coordination = data['coordination']
    coordination_diff = data['coordination_diff']
    significant_changes = data['significant_changes']
    
    # 构型编号（考虑step=5）
    config_nums = np.arange(len(coordination)) * 5 + 1
    
    # 绘制配位数
    ax1 = axes[idx]
    ax1.plot(config_nums, coordination, 'b-', linewidth=2, label='配位数')
    
    # 显式设置字体属性
    ax1.set_xlabel('构型编号', fontsize=12, **font_kwargs)
    ax1.set_ylabel('配位数', fontsize=12, color='b', **font_kwargs)
    ax1.set_title(f'{pair_name} 对的配位数随时间变化', fontsize=14, fontweight='bold', **font_kwargs)
    
    ax1.tick_params(axis='y', labelcolor='b')
    ax1.grid(True, alpha=0.3)
    
    # 标记显著变化点
    if len(significant_changes) > 0:
        ax1.scatter(config_nums[significant_changes], 
                   coordination[significant_changes], 
                   color='red', s=100, zorder=5, label='显著变化点')
        for sc in significant_changes[:5]:  # 只标记前5个
            ax1.axvline(config_nums[sc], color='red', linestyle='--', alpha=0.5)
    
    # 绘制配位数变化率（在右侧y轴）
    ax2 = ax1.twinx()
    ax2.plot(config_nums[1:], coordination_diff, 'g-', alpha=0.5, linewidth=1, label='变化率')
    
    ax2.set_ylabel('配位数变化率', fontsize=12, color='g', **font_kwargs)
    if font_prop and not isinstance(font_prop, str):
        ax1.legend(loc='upper left', prop=font_prop)
        ax2.legend(loc='upper right', prop=font_prop)
    else:
        ax1.legend(loc='upper left')
        ax2.legend(loc='upper right')
    
    ax2.tick_params(axis='y', labelcolor='g')

plt.tight_layout()
plt.savefig('coordination_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


<Figure size 1200x1200 with 6 Axes>

In [9]:
# 计算最短键长随时间的变化（用于进一步分析反应）
def calculate_min_bond_length(configs, elements, n_atoms, elem1_idx, elem2_idx, lattice, step=5):
    """
    计算最短键长随时间的变化
    """
    start_idx1 = sum(n_atoms[:elem1_idx])
    end_idx1 = start_idx1 + n_atoms[elem1_idx]
    
    start_idx2 = sum(n_atoms[:elem2_idx])
    end_idx2 = start_idx2 + n_atoms[elem2_idx]
    
    min_distances = []
    lattice_inv = np.linalg.inv(lattice)
    
    for i in range(0, len(configs), step):
        config = configs[i]
        coords = config['cartesian']
        
        coords1 = coords[start_idx1:end_idx1]
        coords2 = coords[start_idx2:end_idx2]
        
        # 计算所有原子对之间的距离
        distances = []
        for c1 in coords1:
            for c2 in coords2:
                # 计算笛卡尔坐标差
                cart_diff = c1 - c2
                # 转换为分数坐标差
                frac_diff = cart_diff @ lattice_inv
                # 应用周期性边界条件
                frac_diff = frac_diff - np.round(frac_diff)
                # 转换回笛卡尔坐标差（最小镜像约定）
                cart_diff_min = frac_diff @ lattice
                dist = np.linalg.norm(cart_diff_min)
                if dist > 0.01:
                    distances.append(dist)
        
        if len(distances) > 0:
            min_distances.append(min(distances))
        else:
            min_distances.append(0)
    
    return np.array(min_distances)

# 计算最短键长
min_bond_lengths = {}
for pair_name in rdf_results.keys():
    elem1, elem2 = pair_name.split('-')
    elem1_idx = elements.index(elem1)
    elem2_idx = elements.index(elem2)
    
    min_dist = calculate_min_bond_length(configs, elements, n_atoms, 
                                        elem1_idx, elem2_idx, lattice, step=5)
    min_bond_lengths[pair_name] = min_dist
    
    print(f"{pair_name} 对:")
    print(f"  平均最短键长: {np.mean(min_dist):.3f} Å")
    print(f"  最短键长范围: [{np.min(min_dist):.3f}, {np.max(min_dist):.3f}] Å")


Na-S 对:
  平均最短键长: 2.350 Å
  最短键长范围: [1.141, 2.571] Å
Sb-S 对:
  平均最短键长: 2.341 Å
  最短键长范围: [2.135, 2.463] Å
Na-Sb 对:
  平均最短键长: 2.787 Å
  最短键长范围: [1.289, 3.290] Å


In [10]:
# 可视化最短键长随时间的变化
fig, axes = plt.subplots(len(min_bond_lengths), 1, figsize=(12, 4*len(min_bond_lengths)))
if len(min_bond_lengths) == 1:
    axes = [axes]

# 确保使用中文字体
font_prop = get_font_prop()
font_kwargs = get_font_kwargs(font_prop)

for idx, (pair_name, min_dist) in enumerate(min_bond_lengths.items()):
    config_nums = np.arange(len(min_dist)) * 5 + 1
    
    axes[idx].plot(config_nums, min_dist, 'b-', linewidth=2)
    
    # 显式设置字体属性
    axes[idx].set_xlabel('构型编号', fontsize=12, **font_kwargs)
    axes[idx].set_ylabel('最短键长 (Å)', fontsize=12, **font_kwargs)
    axes[idx].set_title(f'{pair_name} 对的最短键长随时间变化', fontsize=14, fontweight='bold', **font_kwargs)
    
    axes[idx].grid(True, alpha=0.3)
    
    # 标记最小值点
    min_idx = np.argmin(min_dist)
    axes[idx].scatter(config_nums[min_idx], min_dist[min_idx], 
                     color='red', s=100, zorder=5, label=f'最小值: {min_dist[min_idx]:.3f} Å')
    
    if font_prop and not isinstance(font_prop, str):
        axes[idx].legend(prop=font_prop)
    else:
        axes[idx].legend()

plt.tight_layout()
plt.savefig('min_bond_length.png', dpi=300, bbox_inches='tight')
plt.show()


<Figure size 1200x1200 with 3 Axes>

## 5. 综合分析：反应时间点总结


In [11]:
# 综合分析所有指标，确定反应发生的时间点
print("="*60)
print("反应时间点综合分析")
print("="*60)

# 收集所有显著变化点
all_change_points = []

for pair_name, data in reaction_times.items():
    if data['first_change'] is not None:
        config_num = data['first_change'] * 5 + 1
        all_change_points.append((pair_name, config_num))
        print(f"\n{pair_name} 对:")
        print(f"  第一个显著变化: 构型 {config_num}")
        print(f"  该时刻配位数: {data['coordination'][data['first_change']]:.2f}")

# 找出最短键长突然减小的点（可能是新键形成）
print("\n最短键长分析:")
for pair_name, min_dist in min_bond_lengths.items():
    # 计算键长的变化率
    dist_diff = np.diff(min_dist)
    # 找到突然减小的点（负值较大）
    threshold = -np.std(dist_diff)
    sudden_decreases = np.where(dist_diff < threshold)[0]
    
    if len(sudden_decreases) > 0:
        first_decrease = sudden_decreases[0]
        config_num = first_decrease * 5 + 1
        print(f"{pair_name} 对:")
        print(f"  键长突然减小发生在: 构型 {config_num}")
        print(f"  键长变化: {min_dist[first_decrease+1] - min_dist[first_decrease]:.3f} Å")

print("\n" + "="*60)
print("建议:")
print("1. 结合RDF变化、配位数变化和最短键长变化综合分析")
print("2. 如果多个指标同时出现显著变化，很可能是反应发生点")
print("3. 可以进一步查看对应构型的原子结构进行验证")
print("="*60)


反应时间点综合分析

Na-S 对:
  第一个显著变化: 构型 1
  该时刻配位数: 2.48

Sb-S 对:
  第一个显著变化: 构型 141
  该时刻配位数: 4.79

Na-Sb 对:
  第一个显著变化: 构型 1
  该时刻配位数: 0.83

最短键长分析:
Na-S 对:
  键长突然减小发生在: 构型 41
  键长变化: -0.067 Å
Sb-S 对:
  键长突然减小发生在: 构型 1
  键长变化: -0.225 Å
Na-Sb 对:
  键长突然减小发生在: 构型 116
  键长变化: -0.088 Å

建议:
1. 结合RDF变化、配位数变化和最短键长变化综合分析
2. 如果多个指标同时出现显著变化，很可能是反应发生点
3. 可以进一步查看对应构型的原子结构进行验证


## 6. 保存分析结果


In [12]:
# 保存RDF数据到文件
import pickle

results = {
    'rdf_results': rdf_results,
    'reaction_times': reaction_times,
    'min_bond_lengths': min_bond_lengths,
    'elements': elements,
    'n_atoms': n_atoms
}

with open('RDF_analysis_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("分析结果已保存到 RDF_analysis_results.pkl")
print("所有图像已保存为PNG文件")


分析结果已保存到 RDF_analysis_results.pkl
所有图像已保存为PNG文件
